In [10]:
#---MED APPT NO SHOW---

import pandas as pd



#--- STEP 2

df = pd.read_csv('MedApptNoSho.csv')

total_column_with_missing = df.isnull().any(axis=0).sum()
print ('columns with missing data: ', total_column_with_missing)
total_rows_with_missing = df.isnull().any(axis=1).sum()
print ('rows with missing data: ', total_rows_with_missing)

#--- STEP 3---
# Define the feature list
features = [
    'Gender',
    'Age',
    'Scholarship',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'SMS_received',
]

# Extract the subset of features
extracted_df = df[features]

# Display the first few rows
print(extracted_df.head())

#---STEP 4---

from sklearn.preprocessing import MinMaxScaler
df_processed = df[features].copy()

# Age has physically impossible Values
df_processed = df_processed[(df_processed['Age'] >= 0) & (df_processed['Age'] <= 110)]

# Convert text 'F' and 'M' into 0 and 1
df_processed['Gender'] = df_processed['Gender'].map({'F': 0, 'M': 1})

# MinMax Scaling normalizes Age into a 0 to 1 range, aligning it with the other features
scaler = MinMaxScaler()
df_processed['Age'] = scaler.fit_transform(df_processed[['Age']])

print(df_processed.head())

#---STEP 5---
from sklearn.model_selection import train_test_split

X = df_processed
y = df['No-show'].loc[X.index]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

criteria = ['gini', 'entropy', 'log_loss']
best_criterion = None
best_validation_accuracy = -1

print("--- Validation Phase (Tuning) ---")
for crit in criteria:

    dt_model = DecisionTreeClassifier(criterion=crit, max_depth=5, random_state=42)
    dt_model.fit(X_train, y_train)

    val_predictions = dt_model.predict(X_val)
    val_accuracy = accuracy_score(y_val, val_predictions)
    print(f"Criterion: {crit:<8} | Validation Accuracy: {val_accuracy:.4f}")
    
    if val_accuracy > best_validation_accuracy:
        best_validation_accuracy = val_accuracy
        best_criterion = crit

print(f"\nSelected Best Criterion: {best_criterion}")

final_classifier = DecisionTreeClassifier(criterion=best_criterion, max_depth=5, random_state=42)
final_classifier.fit(X_train, y_train)

test_predictions = final_classifier.predict(X_test)
final_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final Test Accuracy Score: {final_accuracy:.4f}")

#  confusion matrix
cm = confusion_matrix(y_test, test_predictions, labels=['No', 'Yes'])
print("\nConfusion Matrix:")
print(cm)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Define the number of estimators to test
estimator_options = [10, 100, 300]

for n in estimator_options:
    print(f"\n=========================================")
    print(f"  Random Forest with n_estimators = {n}")
    print(f"=========================================")
    
    # Initialize and train the model
    # Using max_depth=10 to give the forest room to catch interactions
    rf_model = RandomForestClassifier(n_estimators=n, max_depth=10, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    
    # Evaluate on the Test Set
    test_predictions = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, test_predictions)
    
    # Print metrics
    print(f"Test Accuracy Score: {accuracy:.4f}\n")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, test_predictions, labels=['No', 'Yes']))
    print("\nClassification Report:")
    print(classification_report(y_test, test_predictions, target_names=['Show (No)', 'No-Show (Yes)'], zero_division=0))





columns with missing data:  0
rows with missing data:  0
  Gender  Age  Scholarship  Hipertension  Diabetes  Alcoholism  Handcap  \
0      F   62            0             1         0           0        0   
1      M   56            0             0         0           0        0   
2      F   62            0             0         0           0        0   
3      F    8            0             0         0           0        0   
4      F   56            0             1         1           0        0   

   SMS_received  
0             0  
1             0  
2             0  
3             0  
4             0  
   Gender       Age  Scholarship  Hipertension  Diabetes  Alcoholism  Handcap  \
0       0  0.607843            0             1         0           0        0   
1       1  0.549020            0             0         0           0        0   
2       0  0.607843            0             0         0           0        0   
3       0  0.078431            0             0         0    